# Product Pain Points Extraction

## Purpose
This notebook extracts the top 3 keywords from negative reviews (`risk_flag = 1`) for each product. 
Since keyword extraction and regex frequency analysis are difficult to perform efficiently in standard BigQuery SQL, this Python script identifies key customer pain points and writes them back to the `intermediate` layer for business reporting.

In [ ]:
import pandas as pd
import re
from collections import Counter
from google.cloud import bigquery
from datetime import datetime
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables from .env
load_dotenv(find_dotenv())

# Configuration
PROJECT_ID = os.getenv("PROJECT_ID")
INPUT_TABLE = f"{PROJECT_ID}.dbt_ecommerce_staging.stg_reviews"
SENTIMENT_TABLE = f"{PROJECT_ID}.dbt_ecommerce_intermediate.int_review_sentiment_detail"
OUTPUT_TABLE = f"{PROJECT_ID}.dbt_ecommerce_intermediate.int_product_pain_points"

client = bigquery.Client(project=PROJECT_ID)
print(f"[{datetime.now()}] Initialized BigQuery client for project: {PROJECT_ID}")

## Step 1: Load Data from BigQuery
We fetch `product_id` and `review_text` for all reviews flagged as negative (`risk_flag = 1`). We join the staging reviews with the sentiment detail table to get the risk flags.

In [ ]:
print(f"[{datetime.now()}] Fetching negative reviews...")

# Joining stg_reviews with sentiment detail to get review_text and risk_flag
query = f"""
SELECT 
    r.product_id, 
    r.review_text 
FROM `{INPUT_TABLE}` r
JOIN `{SENTIMENT_TABLE}` sd
  ON r.review_id = sd.review_id
WHERE sd.risk_flag = 1
"""

try:
    df_raw = client.query(query).to_dataframe()
    print(f"[{datetime.now()}] Successfully loaded {len(df_raw)} negative reviews.")
except Exception as e:
    print(f"[{datetime.now()}] Error fetching data: {e}")
    df_raw = pd.DataFrame() # Ensure variable exists for next steps

## Step 2: Extract Keywords
Logic:
1. Combine all negative reviews for a product.
2. Extract lowercase English words with length >= 4.
3. Filter out common stop words.
4. Identify the top 3 most frequent words.

In [ ]:
STOP_WORDS = {'this', 'that', 'with', 'from', 'have', 'they', 'what', 'were', 'just', 'very', 'product', 'item'}

def extract_top_pain_points(texts):
    # Combine all review texts for this product
    combined_text = " ".join(str(t) for t in texts if t)
    
    # Find words with length >= 4 using regex
    words = re.findall(r"\b[a-z]{4,}\b", combined_text.lower())
    
    # Filter out stop words
    filtered_words = [w for w in words if w not in STOP_WORDS]
    
    if not filtered_words:
        return "N/A"
    
    # Count frequencies and get top 3
    top_3_counts = Counter(filtered_words).most_common(3)
    top_3_words = [item[0] for item in top_3_counts]
    
    return ", ".join(top_3_words)

print(f"[{datetime.now()}] Processing keywords...")

try:
    if not df_raw.empty:
        # Group by product and apply extraction function
        results = df_raw.groupby("product_id")["review_text"].apply(extract_top_pain_points).reset_index()
        results.columns = ["product_id", "top_pain_points"]
        print(f"[{datetime.now()}] Extraction complete. Found pain points for {len(results)} products.")
    else:
        print("No data to process.")
        results = pd.DataFrame(columns=["product_id", "top_pain_points"])
except Exception as e:
    print(f"[{datetime.now()}] Error during processing: {e}")
    results = pd.DataFrame(columns=["product_id", "top_pain_points"])

## Step 3: Write Results back to BigQuery
We use `WRITE_TRUNCATE` to overwrite the target table with the latest analysis results.

In [ ]:
if not results.empty:
    print(f"[{datetime.now()}] Writing results to {OUTPUT_TABLE}...")
    
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
    )

    try:
        job = client.load_table_from_dataframe(results, OUTPUT_TABLE, job_config=job_config)
        job.result()  # Wait for completion
        print(f"[{datetime.now()}] Successfully uploaded {len(results)} rows to BigQuery.")
    except Exception as e:
        print(f"[{datetime.now()}] Error uploading to BigQuery: {e}")
else:
    print("No results to upload.")